## 1. Data ingestion

In [1]:
import pandas as pd
import numpy as np
import os
import urllib.request
import gzip

def download_geo_matrix(gse_id):
    short_gse_id = gse_id[:5]
    data_url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{short_gse_id}nnn/{gse_id}/matrix/{gse_id}_series_matrix.txt.gz"
    dest_path = os.path.join("data", f"{gse_id}_series_matrix.txt.gz")
    os.makedirs("data", exist_ok=True)
    if not os.path.exists(dest_path):
        print("Downloading dataset from NCBI GEO... This might take a minute.")
        urllib.request.urlretrieve(data_url, dest_path)
        print(f"Download complete! File saved to: {dest_path}")
    else:
        print("Dataset already exists locally in the 'data' folder.")
    return dest_path

file_path = download_geo_matrix("GSE23597")

Dataset already exists locally in the 'data' folder.


## 2. Clinical metadata parsing

GSE23597 shares the GSE16879 series-matrix structure: `!`-prefixed metadata lines, with repeated characteristics keys de-duplicated by counter.

In [2]:
metadata_dict = {}

with gzip.open(file_path, 'rt', encoding='utf8', errors='ignore') as f:
    for line in f:
        clean_line = line.strip()
        if len(clean_line) == 0:
            continue
        elif not clean_line.startswith("!"):
            break

        if clean_line.startswith("!Sample"):
            parts = clean_line.split("\t")
            base_key = parts[0].replace("!Sample_", "")
            values = [v.replace('"', "").strip() for v in parts[1:]]
            final_key = base_key
            counter = 1
            while final_key in metadata_dict:
                final_key = f"{base_key}_{counter}"
                counter += 1
            metadata_dict[final_key] = values

df_clinical_batch2 = pd.DataFrame(metadata_dict)
print(f"Parsed {df_clinical_batch2.shape[0]} samples")

Parsed 113 samples


## 3. Baseline filter

GSE23597 samples each patient at weeks 0, 8 and 30. Only Week 0 precedes treatment; the week 8 and
week 30 profiles are measured after the drug has acted and their expression therefore already
reflects the outcome under prediction. Only Week 0 profiles are retained.

In [3]:
df_clinical_batch2 = df_clinical_batch2.rename(columns={
    'characteristics_ch1_2': 'Time',
    'characteristics_ch1_4': 'Primary Response',
})

# retain Week 0 baseline profiles only
df_baseline = df_clinical_batch2[df_clinical_batch2['Time'] == "time: W0"].copy()

df_baseline['Time'] = df_baseline['Time'].str.replace("time: ", "", regex=False)
df_baseline['Primary Response'] = df_baseline['Primary Response'].str.replace("wk8 response: ", "", regex=False)

print(f"Baseline (W0) patients: {df_baseline.shape[0]}")

Baseline (W0) patients: 45


## 4. Expression matrix and join

An inner join on `geo_accession` aligns clinical labels to the transposed feature matrix, retaining only patients present in both.

In [4]:
df_genes_batch2 = pd.read_csv(
    file_path,
    sep='\t',
    comment="!",
    index_col=0,
    compression='gzip',
)
df_genes_batch2 = df_genes_batch2.T

df_baseline = df_baseline.set_index('geo_accession')
df_final_batch2 = pd.merge(df_baseline, df_genes_batch2,
                           left_index=True, right_index=True, how='inner')

y = df_final_batch2['Primary Response']
X = df_final_batch2.drop(columns=df_baseline.columns)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (45, 54675)
y shape: (45,)


## 5. Class balance

In [5]:
print(y.value_counts())
print(f"\nMajority-class baseline: {y.value_counts().max()}/{len(y)} = {y.value_counts().max()/len(y)*100:.2f}%")

Primary Response
Yes    30
No     15
Name: count, dtype: int64

Majority-class baseline: 30/45 = 66.67%


**Majority-class baseline: 66.67%.** The best constant predictor on this cohort is `Yes`, in contrast
to the discovery cohort, where it is `No` (54.1%). The two cohorts carry opposite response
prevalence.

## 6. Evaluation — training-cohort scaler

The serialised pipeline carries the `StandardScaler` fitted on GSE16879. Applying it to GSE23597
treats the external cohort strictly as unseen data: it is transformed exactly as the training data
was, with no information propagating back from it.

In [6]:
import joblib
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

loaded_model = joblib.load("infliximab_lasso_model.pkl")
print("Classes:", loaded_model.classes_)   # confirms which class occupies index 1

y_binary = (y == 'Yes').astype(int)        # valid only where classes_[1] == 'Yes'

probs_stored = loaded_model.predict_proba(X)[:, 1]
preds_stored = loaded_model.predict(X)

print(pd.Series(preds_stored).value_counts())
print(f"Accuracy: {accuracy_score(y, preds_stored)*100:.2f}%")
print(f"AUC:      {roc_auc_score(y_binary, probs_stored):.3f}")
print(confusion_matrix(y, preds_stored, labels=['No', 'Yes']))

Classes: ['No' 'Yes']
Yes    45
Name: count, dtype: int64
Accuracy: 66.67%
AUC:      0.642
[[ 0 15]
 [ 0 30]]


All 45 patients are assigned `Yes`. The resulting 66.67% accuracy reproduces the majority-class
baseline through a constant prediction, and the `predicted No` column of the confusion matrix is
empty. Accuracy is therefore uninformative on this cohort; AUC, which is threshold-independent, is
the operative metric.

## 7. Input distribution check

Quantifies where the external cohort lands relative to the distribution the classifier was fitted on.
Training data passed through its own scaler yields mean 0 and standard deviation 1 by construction.

In [7]:
X_scaled = loaded_model.named_steps['scaler'].transform(X)
print(f"GSE23597 through the GSE16879 scaler -> mean: {X_scaled.mean():.3f}, sd: {X_scaled.std():.3f}")
print()
print(pd.Series(loaded_model.decision_function(X)).describe())

GSE23597 through the GSE16879 scaler -> mean: 6.566, sd: 5.552

count    45.000000
mean     38.331865
std      16.540748
min       4.247397
25%      26.729350
50%      43.747979
75%      49.542159
max      67.484190
dtype: float64


GSE23597 arrives approximately 6.5 SD from the distribution the model was fitted on, and the decision
function does not fall below +4.25 for any patient — the entire cohort lies on one side of a boundary
at zero.

The mechanism is consistent with independent RMA normalisation of the two series: probe intensities
sit at a different centre, and the resulting offset does not cancel across probes. Summed over 704
non-zero coefficients it dominates the decision function, leaving patient-level variation unable to
reach the boundary. This is covariate shift in the features rather than a label-prior effect — the
collapse is toward `Yes` despite the training cohort being `No`-majority.

## 8. Evaluation — per-cohort standardisation

GSE23597 is standardised against its own mean and variance, bypassing the stored scaler and passing
the result directly to the classifier. This constitutes a location-only batch correction. It operates
on expression values alone and does not access response labels, so the labels remain unseen; it does
relax the strict unseen-data condition of section 6, and is therefore reported alongside that result
rather than in place of it.

In [8]:
from sklearn.preprocessing import StandardScaler

X_own = StandardScaler().fit_transform(X)
clf = loaded_model.named_steps['clf']

preds_own = clf.predict(X_own)
probs_own = clf.predict_proba(X_own)[:, 1]

print(pd.Series(preds_own).value_counts())
print(f"Accuracy: {accuracy_score(y, preds_own)*100:.2f}%")
print(f"AUC:      {roc_auc_score(y_binary, probs_own):.3f}")

Yes    24
No     21
Name: count, dtype: int64
Accuracy: 55.56%
AUC:      0.627


Predictions are mixed (24/21) and AUC moves by 0.015 (0.642 → 0.627): re-centring relocates the
decision threshold without materially altering the model's ranking of patients. Agreement between two
substantially different scaling regimes places ranking performance on this cohort at AUC ≈ 0.63.

Accuracy remains uninformative under both regimes — 66.67% reproduces the baseline via a constant
prediction, and 55.56% falls approximately 11 points below it.

## 9. Bootstrap confidence interval

At n=45 a point estimate of AUC carries wide uncertainty. The interval is estimated by resampling
patients with replacement over 2,000 iterations, discarding degenerate resamples containing a single
class.

In [9]:
rng = np.random.default_rng(2)
yb = y_binary.values
boot = []
for _ in range(2000):
    i = rng.integers(0, len(yb), len(yb))
    if len(np.unique(yb[i])) < 2:
        continue
    boot.append(roc_auc_score(yb[i], probs_own[i]))

print(f"AUC (per-cohort standardisation): {roc_auc_score(yb, probs_own):.3f}")
print(f"95% CI: {np.percentile(boot, 2.5):.3f} - {np.percentile(boot, 97.5):.3f}")

AUC (per-cohort standardisation): 0.627
95% CI: 0.424 - 0.804


## Summary

| | Internal (GSE16879, nested CV) | External (GSE23597) |
|---|---|---|
| n | 61 | 45 |
| Majority-class baseline | 54.1% (`No`) | 66.67% (`Yes`) |
| Accuracy | 68.97% ± 9.13% | 66.67% (constant) / 55.56% (centred) — uninformative |
| AUC | — | 0.627, 95% CI 0.424–0.804 |

The confidence interval contains 0.5: the model's ranking of external patients is not distinguishable
from chance at this sample size. Within the discovery cohort it exceeds the majority-class baseline
by approximately 15 points.

**Limitations.**

* Batch correction beyond section 8 — for example ComBat, which adds per-gene scale correction and
  empirical-Bayes shrinkage — was not pursued. The location-only correction in section 8 moved AUC by
  0.015, and at n=45 with a confidence interval spanning 0.42–0.80 a marginal improvement would not
  be distinguishable from noise.
* GSE16879 and GSE23597 may not define, score or time clinical response identically; GSE23597 scores
  at week 8. This analysis does not separate that effect from covariate shift.
* The binding constraint is sample size: 61 training patients against 54,675 features.
  `02_final_pipeline.ipynb` section 8 quantifies the consequence for feature selection.